In [1]:
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import torch
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM, Conv1D, GlobalMaxPooling1D
import pickle

#### Load data

In [3]:
df = pd.read_csv('../datasets/intents_train.csv')

In [4]:
df.head()

,text,intent
0,Carbon fiber bike,Bike Types
1,budget bikes,Bike Types
2,professional bikes,Bike Types
3,bikes for men,Bike Types
4,bikes for women,Bike Types


In [5]:
df.intent.value_counts()

intent
Return Policy       21
Bike Types          18
Make Appointment    16
Trade-in Options    16
Hours               15
Cost Estimation      7
Welcome Intent       1
Name: count, dtype: int64

In [6]:
test_df = pd.read_csv('../datasets/intents_test.csv')

In [7]:
test_df.shape, df.shape

((37, 1), (94, 2))

#### Константы

In [9]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MAX_LEN = 20
BATCH_SIZE = 8
EPOCHS = 40
LEARNING_RATE = 2e-5
FALLBACK_THRESHOLD = 0.6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [10]:
# переводим в нижний регистр
df['text'] = df['text'].str.lower()

In [11]:
classes = sorted(df['intent'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

#### train test split

In [13]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    [class_to_idx[l] for l in df["intent"]],
    test_size=0.2,
    random_state=RANDOM_STATE
)

#### Токинезация

In [15]:
tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased', force_download=True)

C:\Users\dmitr\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = IntentDataset(train_texts, train_labels, tokenizer_bert, MAX_LEN)
val_dataset   = IntentDataset(val_texts,   val_labels,   tokenizer_bert, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

#### Токенизация для rnn/lstm

In [18]:
vocab_size = 5000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(df['text'])

def prepare_rnn_data(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    X = pad_sequences(sequences, maxlen=MAX_LEN)
    return X

train_X_rnn = prepare_rnn_data(train_texts)
val_X_rnn   = prepare_rnn_data(val_texts)

train_y_rnn = np.array(train_labels)
val_y_rnn   = np.array(val_labels)

print("train_X shape:", train_X_rnn.shape)
print("train_y shape:", train_y_rnn.shape)
print("val_X shape:", val_X_rnn.shape)
print("val_y shape:", val_y_rnn.shape)

train_X shape: (75, 20)
train_y shape: (75,)
val_X shape: (19, 20)
val_y shape: (19,)


### RNN

In [20]:
num_classes = len(np.unique(train_y_rnn))

rnn = Sequential([
    Embedding(input_dim=5000, output_dim=64, mask_zero=True),
    SimpleRNN(32),
    Dense(num_classes, activation='softmax')
])

rnn.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = rnn.fit(
    train_X_rnn,
    train_y_rnn,
    validation_data=(val_X_rnn, val_y_rnn),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

Epoch 1/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.2533 - loss: 1.9252 - val_accuracy: 0.3158 - val_loss: 1.8756
Epoch 2/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6400 - loss: 1.7840 - val_accuracy: 0.4211 - val_loss: 1.8203
Epoch 3/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7467 - loss: 1.6516 - val_accuracy: 0.5789 - val_loss: 1.7586
Epoch 4/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8000 - loss: 1.5027 - val_accuracy: 0.5789 - val_loss: 1.6964
Epoch 5/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8133 - loss: 1.3429 - val_accuracy: 0.5789 - val_loss: 1.6308
Epoch 6/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8400 - loss: 1.1865 - val_accuracy: 0.5789 - val_loss: 1.5611
Epoch 7/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8400 - loss: 1.0500 - val_accuracy: 0.6316 - val_loss: 1.5035
Epoch 8/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8667 - loss: 0.9214 - val_accuracy: 0.6316 - v

In [21]:
y_true = val_y_rnn
y_pred = np.argmax(rnn.predict(val_X_rnn), axis=1)
print("\nClassification report (validation):")
print(classification_report(y_true, y_pred, digits=3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step

Classification report (validation):
              precision    recall  f1-score   support

           0      0.333     0.250     0.286         4
           1      1.000     0.500     0.667         2
           2      0.667     1.000     0.800         4
           3      0.500     0.250     0.333         4
           4      0.600     1.000     0.750         3
           5      0.500     0.500     0.500         2

    accuracy                          0.579        19
   macro avg      0.600     0.583     0.556        19
weighted avg      0.568     0.579     0.540        19



In [22]:
np.argmax(rnn.predict(val_X_rnn), axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


array([0, 2, 3, 4, 4, 2, 2, 4, 3, 0, 5, 2, 4, 4, 2, 0, 1, 2, 5],
      dtype=int64)

### LSTM

In [24]:
num_classes = len(np.unique(train_y_rnn))

lstm = Sequential([
    Embedding(input_dim=5000, output_dim=64, mask_zero=True),
    LSTM(32),
    Dense(num_classes, activation='softmax')
])

lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = lstm.fit(
    train_X_rnn,
    train_y_rnn,
    validation_data=(val_X_rnn, val_y_rnn),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

Epoch 1/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.1200 - loss: 1.9449 - val_accuracy: 0.3684 - val_loss: 1.9330
Epoch 2/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5600 - loss: 1.9191 - val_accuracy: 0.5263 - val_loss: 1.9144
Epoch 3/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 1.8910 - val_accuracy: 0.5263 - val_loss: 1.8917
Epoch 4/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6667 - loss: 1.8525 - val_accuracy: 0.5789 - val_loss: 1.8609
Epoch 5/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6933 - loss: 1.7918 - val_accuracy: 0.5789 - val_loss: 1.8106
Epoch 6/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 1.6974 - val_accuracy: 0.5263 - val_loss: 1.7311
Epoch 7/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6400 - loss: 1.5432 - val_accuracy: 0.4737 - val_loss: 1.6309
Epoch 8/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5867 - loss: 1.3812 - val_accuracy: 0.4737 - v

In [25]:
y_true = val_y_rnn
y_pred = np.argmax(lstm.predict(val_X_rnn), axis=1)
print("\nClassification report (validation):")
print(classification_report(y_true, y_pred, digits=3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step

Classification report (validation):
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         4
           1      0.500     1.000     0.667         2
           2      1.000     0.750     0.857         4
           3      0.667     0.500     0.571         4
           4      0.667     0.667     0.667         3
           5      1.000     1.000     1.000         2

    accuracy                          0.789        19
   macro avg      0.806     0.819     0.794        19
weighted avg      0.825     0.789     0.792        19



### BERT

In [111]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_CLASSES,
    force_download=True
)
model.to(DEVICE)

C:\Users\dmitr\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [112]:
label_counts = Counter(df["intent"])
total = sum(label_counts.values())
class_weights = [total / (NUM_CLASSES * label_counts[c]) for c in classes]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", dict(zip(classes, class_weights)))

Class weights: {'Bike Types': 0.746031746031746, 'Cost Estimation': 1.9183673469387754, 'Hours': 0.8952380952380953, 'Make Appointment': 0.8392857142857143, 'Return Policy': 0.6394557823129252, 'Trade-in Options': 0.8392857142857143, 'Welcome Intent': 13.428571428571429}


In [113]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor, reduction='mean')

C:\Users\dmitr\anaconda3\Lib\site-packages\transformers\optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [114]:
best_val_loss = float("inf")
epochs_without_improvement = 0
PATIENCE = 8

print("Начало обучения...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    @torch.no_grad()
    def run_validation():
        model.eval()
        val_total_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        val_loss = val_total_loss / len(val_loader)
        val_acc = val_correct / val_total
        return val_loss, val_acc, all_preds, all_labels

    val_loss, val_acc, all_preds, all_labels = run_validation()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

    # Ранняя остановка
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Ранняя остановка на эпохе {epoch+1}")
            break

# Отчёт по классам
y_true = [idx_to_class[i] for i in all_labels]
y_pred = [idx_to_class[i] for i in all_preds]
print("\nClassification report (validation):")
print(classification_report(y_true, y_pred, digits=3))

Начало обучения...
Epoch 1/40 | Train Loss: 1.9792 Acc: 0.120 | Val Loss: 1.9785 Acc: 0.053
Epoch 2/40 | Train Loss: 1.9221 Acc: 0.213 | Val Loss: 1.9094 Acc: 0.105
Epoch 3/40 | Train Loss: 1.8282 Acc: 0.227 | Val Loss: 1.7995 Acc: 0.368
Epoch 4/40 | Train Loss: 1.7820 Acc: 0.333 | Val Loss: 1.6539 Acc: 0.368
Epoch 5/40 | Train Loss: 1.6417 Acc: 0.413 | Val Loss: 1.5006 Acc: 0.526
Epoch 6/40 | Train Loss: 1.4677 Acc: 0.613 | Val Loss: 1.3809 Acc: 0.526
Epoch 7/40 | Train Loss: 1.3335 Acc: 0.587 | Val Loss: 1.2238 Acc: 0.789
Epoch 8/40 | Train Loss: 1.1786 Acc: 0.827 | Val Loss: 1.1141 Acc: 0.842
Epoch 9/40 | Train Loss: 1.0238 Acc: 0.920 | Val Loss: 0.9999 Acc: 0.895
Epoch 10/40 | Train Loss: 0.9124 Acc: 0.907 | Val Loss: 0.8769 Acc: 0.842
Epoch 11/40 | Train Loss: 0.7689 Acc: 0.973 | Val Loss: 0.7437 Acc: 0.895
Epoch 12/40 | Train Loss: 0.6141 Acc: 1.000 | Val Loss: 0.6647 Acc: 0.842
Epoch 13/40 | Train Loss: 0.4921 Acc: 0.973 | Val Loss: 0.5355 Acc: 0.947
Epoch 14/40 | Train Loss: 0.

In [115]:
val_loss, val_acc, all_preds, all_labels = run_validation()
y_pred = [idx_to_class[i] for i in all_preds]
all_preds

[3, 1, 3, 4, 0, 2, 2, 4, 0, 1, 5, 2, 5, 4, 0, 0, 1, 2, 3]

#### Оценка и сравнение моделей

In [117]:
result = pd.DataFrame(columns=['Model', 'Params', 'Accuracy', 'F1 Score'])

def evalute_model(model, X_test, y_test, type_name, val_loader, framework='keras'):
    
    if framework == 'transformers':
        val_loss, val_acc, all_preds, all_labels = run_validation()
        y_pred = all_preds
        params = str(model.bert).replace('\n', '').replace(' ', '')
    elif framework == 'keras':
        y_pred = model.predict(X_test)
        y_pred = np.argmax(y_pred, axis=1)
        params = model.get_config()
    else:
        raise ValueError("Unsupported framework")
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    result.loc[len(result)] = [type_name, params, accuracy, f1]
    
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)

In [118]:
evalute_model(rnn, val_X_rnn, val_y_rnn, 'RNN', val_loader, framework='keras')
evalute_model(lstm, val_X_rnn, val_y_rnn, 'LSTM', val_loader, framework='keras')
evalute_model(model, val_X_rnn, val_y_rnn, 'BERT', val_loader, framework='transformers')
result

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Accuracy: 0.5789473684210527
F1 Score: 0.5399749373433584
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Accuracy: 0.7894736842105263
F1 Score: 0.7919799498746867
Accuracy: 0.9473684210526315
F1 Score: 0.9488721804511279


,Model,Params,Accuracy,F1 Score
0,RNN,"{'name': 'sequential', 'trainable': True, 'dty...",0.578947,0.539975
1,LSTM,"{'name': 'sequential_1', 'trainable': True, 'd...",0.789474,0.791980
2,BERT,BertModel((embeddings):BertEmbeddings((word_em...,0.947368,0.948872


In [119]:
# ПРЕДСКАЗАНИЕ С FALLBACK
@torch.no_grad()
def predict_with_fallback(text):
    model.eval()
    encoding = tokenizer_bert(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    ).to(DEVICE)

    outputs = model(**encoding)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1).squeeze()

    max_prob = probs.max().item()
    pred_idx = torch.argmax(probs).item()

    if max_prob < FALLBACK_THRESHOLD:
        return "Fallback Intent"
    else:
        return idx_to_class[pred_idx]

#### Предсказания и сохранение модели

In [121]:
test_df['text'] = test_df['text'].str.lower()

test_dataset = IntentDataset(test_df['text'], [0]*len(test_df), tokenizer_bert, MAX_LEN)
test_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [122]:
test_df['predict'] = test_df['text'].apply(lambda x: predict_with_fallback(x))

In [123]:
FALLBACK_THRESHOLD_LIST = [0.6, 0.5, 0.4, 0.3, 0.2]
for th in FALLBACK_THRESHOLD_LIST:
    FALLBACK_THRESHOLD = th
    test_df['predict'] = test_df['text'].apply(lambda x: predict_with_fallback(x))
    print(f'FALLBACK_THRESHOLD = {FALLBACK_THRESHOLD}')
    print(test_df.predict.value_counts())
    print()

FALLBACK_THRESHOLD = 0.6
predict
Trade-in Options    7
Cost Estimation     7
Make Appointment    5
Bike Types          5
Hours               5
Fallback Intent     4
Return Policy       4
Name: count, dtype: int64

FALLBACK_THRESHOLD = 0.5
predict
Trade-in Options    7
Cost Estimation     7
Hours               6
Make Appointment    5
Return Policy       5
Bike Types          5
Fallback Intent     2
Name: count, dtype: int64

FALLBACK_THRESHOLD = 0.4
predict
Trade-in Options    8
Cost Estimation     7
Make Appointment    6
Hours               6
Return Policy       5
Bike Types          5
Name: count, dtype: int64

FALLBACK_THRESHOLD = 0.3
predict
Trade-in Options    8
Cost Estimation     7
Make Appointment    6
Hours               6
Return Policy       5
Bike Types          5
Name: count, dtype: int64

FALLBACK_THRESHOLD = 0.2
predict
Trade-in Options    8
Cost Estimation     7
Make Appointment    6
Hours               6
Return Policy       5
Bike Types          5
Name: count, dtype: int

In [124]:
FALLBACK_THRESHOLD = 0.5
test_df['predict'] = test_df['text'].apply(lambda x: predict_with_fallback(x))

In [125]:
test_df[['text', 'predict']].to_csv("data/intents.csv", index=False)

In [126]:
test_df

,text,predict,predict_lstm
0,do i need to replace my helmet?,Make Appointment,Make Appointment
1,my bike was stolen what should i do?,Return Policy,Trade-in Options
2,what types or choices of bikes do i have in yo...,Bike Types,Bike Types
3,do you rent bikes?,Trade-in Options,Bike Types
4,i'm a student and i'm doing a project...,Make Appointment,Bike Types
5,can i bring my dog into your shop?,Make Appointment,Make Appointment
6,do you match prices?,Trade-in Options,Fallback Intent
7,can i get an appointment tomorrow?,Make Appointment,Make Appointment
8,how much to repair my bike?,Cost Estimation,Cost Estimation
9,how do i know what size tire/tube i need?,Cost Estimation,Return Policy


In [127]:
test_texts = test_df['text'].tolist()
test_X_rnn = prepare_rnn_data(test_texts)

In [128]:
def fallback_lstm():
    probs = lstm.predict(test_X_rnn)
    max_prob = probs.max(axis=1)
    pred_idx = np.argmax(lstm.predict(test_X_rnn), axis=1)

    res = [idx_to_class[pred] if prob > FALLBACK_THRESHOLD else "Fallback Intent" for prob, pred in zip(max_prob, pred_idx)]
    return res


test_df['predict_lstm'] = fallback_lstm()

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


In [129]:
FALLBACK_THRESHOLD_LIST = [0.6, 0.5, 0.4, 0.3, 0.2]
for th in FALLBACK_THRESHOLD_LIST:
    FALLBACK_THRESHOLD = th
    test_df['predict_lstm'] = fallback_lstm()
    print(f'FALLBACK_THRESHOLD = {FALLBACK_THRESHOLD}')
    print(test_df.predict_lstm.value_counts())
    print()

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
FALLBACK_THRESHOLD = 0.6
predict_lstm
Fallback Intent     10
Make Appointment     6
Return Policy        6
Bike Types           5
Cost Estimation      4
Trade-in Options     3
Hours                3
Name: count, dtype: int64

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
FALLBACK_THRESHOLD = 0.5
predict_lstm
Fallback Intent     7
Return Policy       7
Make Appointment    6
Trade-in Options    5
Bike Types          5
Cost Estimation     4
Hours               3
Name: count, dtype: int64

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
FALLBACK_THRESHOLD = 0.4
predict_lstm
Return Policy       7
Make Appointment    6
Trade-in Options    6
Bike Types          6
Fallback Intent     5
Cost Estimation     4
Hours               3
Name: count, dtype: int64

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
FALLBACK_THRESHOLD = 0.3
predict

In [130]:
FALLBACK_THRESHOLD = 0.3
test_df['predict_lstm'] = fallback_lstm()
test_df

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


,text,predict,predict_lstm
0,do i need to replace my helmet?,Make Appointment,Make Appointment
1,my bike was stolen what should i do?,Return Policy,Trade-in Options
2,what types or choices of bikes do i have in yo...,Bike Types,Bike Types
3,do you rent bikes?,Trade-in Options,Bike Types
4,i'm a student and i'm doing a project...,Make Appointment,Bike Types
5,can i bring my dog into your shop?,Make Appointment,Make Appointment
6,do you match prices?,Trade-in Options,Fallback Intent
7,can i get an appointment tomorrow?,Make Appointment,Make Appointment
8,how much to repair my bike?,Cost Estimation,Cost Estimation
9,how do i know what size tire/tube i need?,Cost Estimation,Return Policy


In [131]:
# Pytorch
torch.save(model.state_dict(), "data/bert_intent_model.pt")

# Pickle
with open("data/model.pkl", "wb") as f:
    pickle.dump(model.state_dict(), f)

# оба файла весят по 430 МБ каждый. В репо не влезут.

In [132]:
# Загрузка
'''
from transformers import BertForSequenceClassification

num_labels = NUM_CLASSES #7
model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=num_labels
)
model.load_state_dict(torch.load(data/bert_intent_model.pt", map_location="cpu"))
model.eval()
'''

'\nfrom transformers import BertForSequenceClassification\n\nnum_labels = NUM_CLASSES #7\nmodel = BertForSequenceClassification.from_pretrained(\n    "bert-base-multilingual-cased",\n    num_labels=num_labels\n)\nmodel.load_state_dict(torch.load(data/bert_intent_model.pt", map_location="cpu"))\nmodel.eval()\n'

### CNN

In [134]:
cnn = Sequential([
    Embedding(input_dim=5000, output_dim=64, mask_zero=True),
    Conv1D(32, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(num_classes, activation='softmax')
])

cnn.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy'])

history = cnn.fit(
    train_X_rnn,
    train_y_rnn,
    validation_data=(val_X_rnn, val_y_rnn),
    epochs=40,
    batch_size=BATCH_SIZE,
    verbose=1
)

evalute_model(cnn, val_X_rnn, val_y_rnn, 'CNN', val_loader, framework='keras')

Epoch 1/40


C:\Users\dmitr\anaconda3\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'conv1d_1' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1467 - loss: 1.9254 - val_accuracy: 0.2105 - val_loss: 1.9003
Epoch 2/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2400 - loss: 1.8649 - val_accuracy: 0.3158 - val_loss: 1.8783
Epoch 3/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3067 - loss: 1.8173 - val_accuracy: 0.3158 - val_loss: 1.8509
Epoch 4/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3333 - loss: 1.7712 - val_accuracy: 0.2105 - val_loss: 1.8276
Epoch 5/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3067 - loss: 1.7264 - val_accuracy: 0.2105 - val_loss: 1.8008
Epoch 6/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3067 - loss: 1.6790 - val_accuracy: 0.2105 - val_loss: 1.7724
Epoch 7/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3200 - loss: 1.6299 - val_accuracy: 0.2105 - val_loss: 1.7431
Epoch 8/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4267 - loss: 1.5801 - val_accuracy: 0.2105 - val_loss: 1.7085
Ep

In [135]:
result

,Model,Params,Accuracy,F1 Score
0,RNN,"{'name': 'sequential', 'trainable': True, 'dty...",0.578947,0.539975
1,LSTM,"{'name': 'sequential_1', 'trainable': True, 'd...",0.789474,0.791980
2,BERT,BertModel((embeddings):BertEmbeddings((word_em...,0.947368,0.948872
3,CNN,"{'name': 'sequential_3', 'trainable': True, 'd...",0.842105,0.838596
